<a href="https://colab.research.google.com/github/chirayu08/Brain-Tumor-Detection-from-MRI-using-CNN-LSTM/blob/notebooks/Brain_Tumor_Detection_CNN_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Cell 1: Setup and GPU configuration

import subprocess
import tensorflow as tf
import numpy as np
import random

# Check GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Configure GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Set random seeds
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

Mon Mar  9 23:17:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Cell 2: Install dependencies and import libraries

!pip install -q numpy pandas matplotlib seaborn scikit-learn opencv-python tqdm

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score
)

import cv2
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Dropout,
    Dense, Flatten, BatchNormalization,
    GlobalAveragePooling2D, Concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint,
    ReduceLROnPlateau, LearningRateScheduler
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

In [6]:
# Cell 3: Download dataset from Kaggle

from google.colab import files
import os

# Upload kaggle.json
uploaded = files.upload()

# Configure Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d ahmedhamada0/brain-tumor-detection -p /content

# Extract dataset
!rm -rf /content/dataset
!unzip -q /content/brain-tumor-detection.zip -d /content/dataset

# Verify dataset
tumor_files = len([f for f in os.listdir('/content/dataset/yes') if f.endswith(('.jpg','.jpeg','.png'))])
healthy_files = len([f for f in os.listdir('/content/dataset/no') if f.endswith(('.jpg','.jpeg','.png'))])

print("Tumor images:", tumor_files)
print("Healthy images:", healthy_files)
print("Total images:", tumor_files + healthy_files)

Saving kaggle.json to kaggle (5).json
Dataset URL: https://www.kaggle.com/datasets/ahmedhamada0/brain-tumor-detection
License(s): copyright-authors
brain-tumor-detection.zip: Skipping, found more recently modified local copy (use --force to force download)
Tumor images: 1500
Healthy images: 1500
Total images: 3000


In [ ]:
# Cell 4: Data loading and preprocessing

IMG_SIZE = 150

def advanced_preprocess(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    img = cv2.fastNlMeansDenoising(img, h=10)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LANCZOS4)
    img = img.astype("float32") / 255.0

    return img


images = []
labels = []

tumor_path = "/content/dataset/yes"
tumor_files = sorted([f for f in os.listdir(tumor_path) if f.endswith((".jpg", ".jpeg", ".png"))])

for file in tqdm(tumor_files):
    img = advanced_preprocess(os.path.join(tumor_path, file))
    if img is not None:
        images.append(img)
        labels.append(0)


healthy_path = "/content/dataset/no"
healthy_files = sorted([f for f in os.listdir(healthy_path) if f.endswith((".jpg", ".jpeg", ".png"))])

for file in tqdm(healthy_files):
    img = advanced_preprocess(os.path.join(healthy_path, file))
    if img is not None:
        images.append(img)
        labels.append(1)


X = np.array(images, dtype="float32")
y = np.array(labels, dtype="int32")

X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y_categorical = to_categorical(y, num_classes=2)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical,
    test_size=0.15,
    random_state=SEED,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,
    random_state=SEED,
    stratify=np.argmax(y_train, axis=1)
)

 63%|██████▎   | 948/1500 [02:10<00:47, 11.67it/s]

In [ ]:
# Cell 5: Build CNN model

def build_advanced_cnn_model(input_shape=(150,150,1), num_classes=2):

    inputs = Input(shape=input_shape)

    x = Conv2D(64,(3,3),padding="same",activation="relu")(inputs)
    x = BatchNormalization()(x)
    x = Conv2D(64,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.2)(x)

    x = Conv2D(128,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = Conv2D(128,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.25)(x)

    x = Conv2D(256,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = Conv2D(256,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.3)(x)

    x = Conv2D(512,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = Conv2D(512,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.35)(x)

    x = Conv2D(512,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = Conv2D(512,(3,3),padding="same",activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.4)(x)

    x = GlobalAveragePooling2D()(x)

    x = Dense(512,activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    x = Dense(256,activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    outputs = Dense(num_classes,activation="softmax")(x)

    model = Model(inputs,outputs)
    return model


model = build_advanced_cnn_model(input_shape=(IMG_SIZE,IMG_SIZE,1), num_classes=2)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

model.summary()

In [ ]:
# Cell 6: Data augmentation

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest",
    brightness_range=[0.85, 1.15]
)

val_datagen = ImageDataGenerator()

In [ ]:
# Cell 7: Train model

os.makedirs("/content/outputs", exist_ok=True)

def lr_schedule(epoch, lr):
    if epoch > 0 and epoch % 10 == 0:
        return lr * 0.5
    return lr


callbacks = [
    ModelCheckpoint(
        filepath="/content/outputs/best_model.h5",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        min_delta=0.0001
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        min_delta=0.0001
    ),

    LearningRateScheduler(lr_schedule)
]


EPOCHS = 60
BATCH_SIZE = 32

history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True),
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=callbacks,
    steps_per_epoch=len(X_train) // BATCH_SIZE
)

model.save("/content/outputs/final_model.h5")

In [ ]:
# Cell 8: Model evaluation

best_model = keras.models.load_model("/content/outputs/best_model.h5")

y_pred_proba = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba, average="weighted", multi_class="ovr")
except:
    roc_auc = roc_auc_score(y_true, y_pred_proba[:,1])

precision_per_class = precision_score(y_true, y_pred, average=None)
recall_per_class = recall_score(y_true, y_pred, average=None)
f1_per_class = f1_score(y_true, y_pred, average=None)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

target_names = ["Brain Tumor", "Healthy"]
print(classification_report(y_true, y_pred, target_names=target_names))

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

metrics_df.to_csv("/content/outputs/test_metrics.csv", index=False)

In [ ]:
# Cell 9: Visualizations

plt.style.use("seaborn-v0_8-darkgrid")

# Training history
fig, axes = plt.subplots(2, 3, figsize=(18,12))

axes[0,0].plot(history.history["accuracy"])
axes[0,0].plot(history.history["val_accuracy"])
axes[0,0].set_title("Accuracy")

axes[0,1].plot(history.history["loss"])
axes[0,1].plot(history.history["val_loss"])
axes[0,1].set_title("Loss")

axes[0,2].plot(history.history["precision"])
axes[0,2].plot(history.history["val_precision"])
axes[0,2].set_title("Precision")

axes[1,0].plot(history.history["recall"])
axes[1,0].plot(history.history["val_recall"])
axes[1,0].set_title("Recall")

axes[1,1].plot(history.history["auc"])
axes[1,1].plot(history.history["val_auc"])
axes[1,1].set_title("AUC")

if "lr" in history.history:
    axes[1,2].plot(history.history["lr"])
    axes[1,2].set_title("Learning Rate")
else:
    axes[1,2].axis("off")

plt.tight_layout()
plt.savefig("/content/outputs/training_history.png", dpi=300)
plt.show()


# Confusion matrix
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Brain Tumor","Healthy"],
            yticklabels=["Brain Tumor","Healthy"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig("/content/outputs/confusion_matrix.png", dpi=300)
plt.show()


# ROC curve
fpr, tpr, _ = roc_curve(y_test[:,1], y_pred_proba[:,1])
roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_value:.4f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.savefig("/content/outputs/roc_curve.png", dpi=300)
plt.show()


# Sample predictions
num_samples = 16
indices = np.random.choice(len(X_test), num_samples, replace=False)

fig, axes = plt.subplots(4,4, figsize=(12,12))
class_names = ["Brain Tumor","Healthy"]

for idx, ax in enumerate(axes.flat):
    i = indices[idx]
    ax.imshow(X_test[i].squeeze(), cmap="gray")
    ax.set_title(f"T:{class_names[y_true[i]]} P:{class_names[y_pred[i]]}")
    ax.axis("off")

plt.tight_layout()
plt.savefig("/content/outputs/sample_predictions.png", dpi=300)
plt.show()

In [ ]:
# Cell 10: Display results and download

from IPython.display import Image, display
import shutil
from google.colab import files

metrics_display = pd.read_csv("/content/outputs/test_metrics.csv")
print(metrics_display)

display(Image("/content/outputs/training_history.png"))
display(Image("/content/outputs/confusion_matrix.png"))
display(Image("/content/outputs/roc_curve.png"))
display(Image("/content/outputs/sample_predictions.png"))

shutil.make_archive("/content/brain_tumor_results", "zip", "/content/outputs")

files.download("/content/brain_tumor_results.zip")